# Precompute curriculum SEARCH/REPLACE corruption

Generates a fixed set of pre-corrupted (enc_ids, dec_ids) pairs for the pretrain
curriculum instead of corrupting on the fly in `PretrainDataset.__getitem__`.

Local/CPU-only -- no GPU needed, run this on your own machine (not Colab) so it
doesn't cost compute units. Generates `N_VARIANTS_TRAIN` distinct corruptions per
train sample (cycled per epoch via `PrecomputedPretrainDataset.set_variant`, to
keep most of the on-the-fly version's per-epoch augmentation diversity) and a
single deterministic variant for val (a fixed validation corruption is arguably
*more* correct anyway -- it stops the val-loss curve from being noisy purely
because a different random mask was drawn each epoch).

Output: `data/curriculum_precomputed/{train,val}/tierN/variant_V.jsonl`, each line
`{"enc_ids": [...], "dec_ids": [...]}`. Upload this folder to Drive alongside
`data/curriculum/` to use it from the Colab training loop (see the loader class
`src.data_loaders.PrecomputedPretrainDataset` for the swap-in).

In [1]:
import sys, json, time, random
sys.path.insert(0, r".")
from pathlib import Path

from src.model_config import ModelConfig
from src.tokenizer import TLM_Tokenizer
from src.data_set import iter_raw_code, make_sr_corruption

N_VARIANTS_TRAIN = 5
MAX_RETRIES = 4          # retries on the SAME code, matching PretrainDataset._try
MAX_FALLBACK_GUARD = 32  # then tries OTHER random codes, matching PretrainDataset.__getitem__
OUT_ROOT = Path('../data/curriculum_precomputed_fixed')
TIER_PATHS = [Path(f'../data/curriculum/tier{i}.jsonl') for i in range(1, 6)]

config = ModelConfig()
tok = TLM_Tokenizer(vocab_size=config.vocab_size)
tok.load('../src/tokenizer/tokenizer_tlm_small.json')
print(f"vocab_size = {tok.tokenizer.get_vocab_size()}")

[Config] Estimated params: embed=1920000, enc_attn=589824, dec_attn=1179648, enc_ffn=1183488, dec_ffn=1183488, enc_ln=3456, dec_ln=4992, task_emb=4608, copy_head=193, head=0
ModelConfig [TLM_S]
  d_model=192, n_heads=4
  Encoder: 4L  Decoder: 4L  FFN: 768
  Estimated: 6.07M params  (target: 5.00M)
  Vocab: 10000  MaxSeq: 1024  TaskTypes: 6  WeightTie: True
[Config] Estimated params: embed=1920000, enc_attn=589824, dec_attn=1179648, enc_ffn=1183488, dec_ffn=1183488, enc_ln=3456, dec_ln=4992, task_emb=4608, copy_head=193, head=0
Estimated params: 6.07M
vocab_size = 10000


In [2]:
def _try(code, max_retries=MAX_RETRIES):
    for _ in range(max_retries):
        res = make_sr_corruption(code, tok, config)
        if res is not None:
            return res
    return None


def corrupt_with_fallback(codes, idx):
    """Mirrors PretrainDataset.__getitem__: retry same idx, then fall back to
    other random indices in the same tier, before ever emitting a degenerate pair."""
    res = _try(codes[idx])
    guard = 0
    while res is None and guard < MAX_FALLBACK_GUARD:
        idx = random.randrange(len(codes))
        res = _try(codes[idx])
        guard += 1
    return res


def generate_split(split_name, n_variants):
    for tier_path in TIER_PATHS:
        tier_no = tier_path.stem.replace('tier', '')
        codes = list(iter_raw_code(tier_path, split_name=split_name))
        tier_dir = OUT_ROOT / split_name / f'tier{tier_no}'
        tier_dir.mkdir(parents=True, exist_ok=True)

        for v in range(n_variants):
            out_path = tier_dir / f'variant_{v}.jsonl'
            t0 = time.time()
            n_ok, n_fallback = 0, 0
            with out_path.open('w', encoding='utf-8') as f:
                for i in range(len(codes)):
                    res = corrupt_with_fallback(codes, i)
                    if res is None:
                        # degenerate guarantee -- should be ~never hit, matches
                        # PretrainDataset's own last-resort guarantee
                        bos, eos = config.special_token_ids['bos'], config.special_token_ids['eos']
                        res = ([bos, eos], [bos, eos])
                        n_fallback += 1
                    else:
                        n_ok += 1
                    enc_ids, dec_ids = res
                    f.write(json.dumps({'enc_ids': enc_ids, 'dec_ids': dec_ids}) + '\n')
            dt = time.time() - t0
            print(f'{split_name}/tier{tier_no}/variant_{v}: {len(codes)} samples '
                  f'({n_ok} ok, {n_fallback} fallback) in {dt:.1f}s')

In [3]:
t_start = time.time()
print('Generating TRAIN variants...')
generate_split('train', N_VARIANTS_TRAIN)
print('Generating VAL variant (single, deterministic)...')
generate_split('val', 1)
print(f'Done in {(time.time() - t_start)/60:.1f} min total')

Generating TRAIN variants...
train/tier1/variant_0: 35307 samples (35307 ok, 0 fallback) in 64.2s
train/tier1/variant_1: 35307 samples (35307 ok, 0 fallback) in 62.3s
train/tier1/variant_2: 35307 samples (35307 ok, 0 fallback) in 65.1s
train/tier1/variant_3: 35307 samples (35307 ok, 0 fallback) in 64.6s
train/tier1/variant_4: 35307 samples (35307 ok, 0 fallback) in 101.9s
train/tier2/variant_0: 35370 samples (35370 ok, 0 fallback) in 259.1s
train/tier2/variant_1: 35370 samples (35370 ok, 0 fallback) in 132.2s
train/tier2/variant_2: 35370 samples (35370 ok, 0 fallback) in 133.0s
train/tier2/variant_3: 35370 samples (35370 ok, 0 fallback) in 131.2s
train/tier2/variant_4: 35370 samples (35370 ok, 0 fallback) in 135.2s
train/tier3/variant_0: 35057 samples (35057 ok, 0 fallback) in 198.4s
train/tier3/variant_1: 35057 samples (35057 ok, 0 fallback) in 149.1s
train/tier3/variant_2: 35057 samples (35057 ok, 0 fallback) in 147.6s
train/tier3/variant_3: 35057 samples (35057 ok, 0 fallback) in 14

## Sanity check -- load back through the drop-in dataset class

In [4]:
from src.data_loaders import load_precomputed_tiers

train_precomputed = load_precomputed_tiers(OUT_ROOT / 'train', tiers=[1, 2, 3, 4, 5], n_variants=N_VARIANTS_TRAIN)
val_precomputed   = load_precomputed_tiers(OUT_ROOT / 'val',   tiers=[1, 2, 3, 4, 5], n_variants=1)

for t, ds in train_precomputed.items():
    print(f'tier{t}: {len(ds)} samples x {ds.n_variants} variants')

_ds = train_precomputed[1]
_enc, _dec = _ds[0]
print('\nsample 0, variant 0:', tok.decode(_enc, debug=True)[:80])
_ds.set_variant(1)
_enc2, _dec2 = _ds[0]
print('sample 0, variant 1:', tok.decode(_enc2, debug=True)[:80])
print('\nvariants differ:', _enc != _enc2 or _dec != _dec2)

JSONDecodeError: Extra data: line 1 column 4 (char 3)